# 🧠 Customer Cluster Profiling Analysis

This notebook analyzes K-Means customer clusters using RFM (Recency, Frequency, Monetary) data to:
- Understand what each cluster represents
- Identify customer behavior patterns
- Generate actionable business insights for each segment

**Key Question:** What makes each cluster unique?

## Section 1: Import Required Libraries

Import NumPy, Pandas, Scikit-learn, Matplotlib, and Seaborn for data manipulation, clustering, and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Section 2: Load and Prepare RFM Data

Load the RFM dataset from training artifacts and prepare for cluster profiling analysis.

In [ ]:
# 🔥 Load training data from artifacts
print("📂 Loading training data from artifacts...")

train_df = pd.read_csv("../../../artifacts/train.csv")
kmeans_model = joblib.load("../../../artifacts/kmeans.pkl")

print(f"✅ Loaded {len(train_df)} training records")
print(f"✅ Loaded KMeans model with {kmeans_model.n_clusters} clusters")

# Display dataset info
print("\n📊 Dataset Overview:")
print(train_df.head())
print(f"\nShape: {train_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")

## Section 3: Profile Clusters with Aggregate Statistics

Analyze the mean characteristics of each cluster to understand their behavioral patterns.

In [ ]:
# 🔥 CREATE CLUSTER PROFILE
print("📊 Computing Cluster Profiles...\n")

# Select key features for profiling
key_features = [
    "frequency_log", "monetary_log", "tenure", "purchase_rate", 
    "monetary_per_day", "avg_order_value", "unique_items_purchased", "Churn_Label"
]

# Calculate mean values for each cluster
cluster_profile = train_df.groupby("final_kmeans_cluster")[key_features].mean()

print("📈 Cluster Profile (Mean Values):")
print("=" * 100)
print(cluster_profile.round(3))
print("=" * 100)

# More detailed view with counts
print("\n👥 Cluster Size:")
cluster_sizes = train_df["final_kmeans_cluster"].value_counts().sort_index()
for cluster, size in cluster_sizes.items():
    pct = size / len(train_df) * 100
    print(f"  Cluster {cluster}: {size} customers ({pct:.1f}%)")

## Section 4: Interpret Cluster Characteristics

Analyze and label each cluster based on their statistical profiles to understand what they represent.

In [ ]:
# 🧠 INTERPRET CLUSTER MEANINGS
print("\n🎯 CLUSTER INTERPRETATION:\n")

for cluster in sorted(train_df["final_kmeans_cluster"].unique()):
    cluster_data = train_df[train_df["final_kmeans_cluster"] == cluster]
    
    # Get metrics
    avg_freq = cluster_data["frequency_log"].mean()
    avg_monetary = cluster_data["monetary_log"].mean()
    avg_tenure = cluster_data["tenure"].mean()
    avg_churn = cluster_data["Churn_Label"].mean()
    
    print(f"📌 Cluster {cluster}:")
    print(f"   • Size: {len(cluster_data)} customers ({len(cluster_data)/len(train_df)*100:.1f}%)")
    print(f"   • Avg Frequency (log): {avg_freq:.2f}")
    print(f"   • Avg Monetary (log): {avg_monetary:.2f}")
    print(f"   • Avg Tenure: {avg_tenure:.1f} days")
    print(f"   • Churn Rate: {avg_churn*100:.1f}%")
    
    # Label the cluster
    if avg_monetary > train_df["monetary_log"].mean() and avg_churn < train_df["Churn_Label"].mean():
        label = "💎 PREMIUM/LOYAL CUSTOMERS"
    elif avg_freq > train_df["frequency_log"].mean():
        label = "🛍️ FREQUENT BUYERS"
    elif avg_churn > train_df["Churn_Label"].mean():
        label = "⚠️ AT-RISK SEGMENT"
    else:
        label = "📊 LOW ENGAGEMENT USERS"
    
    print(f"   → Label: {label}\n")

## Section 5: Visualize Cluster Profiles

Create heatmaps and bar charts to compare cluster characteristics visually.

In [ ]:
# 📊 HEATMAP OF CLUSTER PROFILES
fig, ax = plt.subplots(figsize=(14, 6))

# Normalize profiles for better visualization
cluster_profile_norm = cluster_profile.div(cluster_profile.max(axis=0), axis=1)

sns.heatmap(cluster_profile_norm.T, annot=True, cmap="YlOrRd", fmt=".2f", 
            cbar_kws={"label": "Normalized Value"}, ax=ax)
ax.set_title("🔥 Cluster Profile Heatmap (Normalized)", fontsize=14, fontweight='bold')
ax.set_xlabel("Cluster", fontsize=12)
ax.set_ylabel("Features", fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("✅ Heatmap shows normalized values (0-1) across clusters for easy comparison")

In [ ]:
# 📈 BAR CHARTS FOR KEY METRICS
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Churn Rate by Cluster
churn_by_cluster = train_df.groupby("final_kmeans_cluster")["Churn_Label"].agg(["mean", "count"])
axes[0, 0].bar(churn_by_cluster.index, churn_by_cluster["mean"]*100, color="coral")
axes[0, 0].set_title("Churn Rate by Cluster", fontweight='bold')
axes[0, 0].set_ylabel("Churn Rate (%)")
axes[0, 0].set_xlabel("Cluster")

# Monetary Value by Cluster
monetary_by_cluster = train_df.groupby("final_kmeans_cluster")["monetary_log"].mean()
axes[0, 1].bar(monetary_by_cluster.index, monetary_by_cluster, color="green")
axes[0, 1].set_title("Avg Monetary Value (log) by Cluster", fontweight='bold')
axes[0, 1].set_ylabel("Monetary (log)")
axes[0, 1].set_xlabel("Cluster")

# Purchase Frequency by Cluster
freq_by_cluster = train_df.groupby("final_kmeans_cluster")["frequency_log"].mean()
axes[1, 0].bar(freq_by_cluster.index, freq_by_cluster, color="blue")
axes[1, 0].set_title("Avg Purchase Frequency (log) by Cluster", fontweight='bold')
axes[1, 0].set_ylabel("Frequency (log)")
axes[1, 0].set_xlabel("Cluster")

# Cluster Size
cluster_sizes = train_df["final_kmeans_cluster"].value_counts().sort_index()
axes[1, 1].bar(cluster_sizes.index, cluster_sizes.values, color="purple")
axes[1, 1].set_title("Cluster Size Distribution", fontweight='bold')
axes[1, 1].set_ylabel("Number of Customers")
axes[1, 1].set_xlabel("Cluster")

plt.tight_layout()
plt.show()

## Section 6: Generate Business Insights and Recommendations

Develop actionable recommendations for each cluster based on their profiling data.

In [ ]:
# 🎯 ACTIONABLE BUSINESS RECOMMENDATIONS
print("\n" + "="*100)
print("🎯 BUSINESS INSIGHTS & RECOMMENDATIONS FOR EACH CLUSTER")
print("="*100 + "\n")

for cluster in sorted(train_df["final_kmeans_cluster"].unique()):
    cluster_data = train_df[train_df["final_kmeans_cluster"] == cluster]
    
    avg_freq = cluster_data["frequency_log"].mean()
    avg_monetary = cluster_data["monetary_log"].mean()
    avg_tenure = cluster_data["tenure"].mean()
    avg_churn = cluster_data["Churn_Label"].mean()
    
    freq_rank = "High" if avg_freq > train_df["frequency_log"].mean() else "Low"
    monetary_rank = "High" if avg_monetary > train_df["monetary_log"].mean() else "Low"
    churn_rank = "High" if avg_churn > train_df["Churn_Label"].mean() else "Low"
    
    print(f"\n📌 CLUSTER {cluster}")
    print(f"   Profile: {freq_rank} Frequency | {monetary_rank} Monetary | {churn_rank} Churn Rate")
    print(f"   Size: {len(cluster_data)} customers ({len(cluster_data)/len(train_df)*100:.1f}%)\n")
    
    print(f"   🎯 Recommendations:")
    
    if monetary_rank == "High" and churn_rank == "Low":
        print("   • VIP/LOYAL SEGMENT - Focus on retention")
        print("   • Offer exclusive loyalty benefits and early access to new products")
        print("   • Implement dedicated account management")
        print("   • Personalized communication for premium experience")
    elif freq_rank == "High":
        print("   • FREQUENT BUYERS - Increase AOV (Average Order Value)")
        print("   • Cross-sell and upsell strategies")
        print("   • Reward loyalty with tiered benefits")
        print("   • Bundle offerings to increase transaction value")
    elif churn_rank == "High":
        print("   • AT-RISK SEGMENT - Churn prevention needed")
        print("   • Win-back campaigns with special offers")
        print("   • Feedback collection to understand churn reasons")
        print("   • Personalized re-engagement initiatives")
    else:
        print("   • LOW ENGAGEMENT - Activation strategies")
        print("   • Re-introduction campaigns with attractive offers")
        print("   • Simplified purchasing process")
        print("   • Community/social engagement initiatives")

print("\n" + "="*100)

## Summary

### Key Takeaways:

✅ **Cluster Profiling Complete** - We now understand what each cluster represents  
✅ **Business Labels Assigned** - Each segment has a clear business meaning  
✅ **Actionable Insights Generated** - Specific strategies for each cluster  
✅ **Ready for Implementation** - Use these insights in your customer segmentation strategy  

### Next Steps:

1. Use cluster assignments in your Streamlit app (✅ Already showing!)
2. Implement cluster-specific marketing campaigns
3. Monitor cluster migration (do customers move clusters over time?)
4. A/B test recommendations for each cluster
5. Refine cluster definitions based on business results

### Interview-Ready Answer:

> "I performed customer segmentation using K-Means clustering on RFM (Recency, Frequency, Monetary) data. By analyzing cluster profiles, I identified distinct customer segments: high-value loyalists, frequent buyers, at-risk customers, and low-engagement users. Each cluster received tailored business recommendations for targeted retention and growth strategies."